### Bridge DAS Processing Part I

#### Step 1: Load Bridge DAS Data

In [ ]:
# Import necessary dependencies
import sys
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy import signal

sys.path.append('..')

from src.ani import bandpass_filter_tukey
from src.plot import plot_das_wavefield, plot_das_psd

In [ ]:
# 1. Define paths 
bridge_dir = Path('..') / 'data' / 'raw_bridge'

# Grab the first processed bridge files
bridge_files = sorted(bridge_dir.glob('*_bridge.npz'))
das_paths = bridge_files[:1]

if not das_paths:
    raise FileNotFoundError(f"No _bridge.npz files found in {bridge_dir}")

for p in das_paths:
    print(f"Found: {p.name}")

In [ ]:
# 2. Load the first file
das_data = np.load(das_paths[0])

print("\nKeys inside .npz file:")
for key in das_data:
    print(f' - {key}')

In [ ]:
# 3. Extract data and natively saved metadata
raw_array = das_data['data']
dt = float(das_data['dt'])           # Natively extracted (0.004s)
dx = float(das_data['dx'])           # Natively extracted (2.45m)
fs = 1.0 / dt

N = raw_array.shape[1]               # Number of time samples
num_channels = raw_array.shape[0]    # Number of channels

In [ ]:
# 4. PREPROCESSING: Zero-mean the data (DC Offset Removal)
das_array = raw_array 

# 5. Create t_axis and x_axis ON THE FLY
t_axis = np.arange(N) * dt             # Time in seconds
x_axis = np.arange(num_channels) * dx  # Distance in METERS for bridge scale

print(f"\nDAS array shape: {das_array.shape} (Channels x Time)")
print(f"Time axis shape: {t_axis.shape}")
print(f"Channel axis shape: {x_axis.shape}")
print(f"Total duration: {(N * dt) / 60:.2f} minutes")
print(f"Total bridge length: {x_axis[-1]:.2f} meters")

In [ ]:
# 6. Plotting
# Compute clipping value to avoid extreme outliers in the colormap
pclip = 99
das_clip = np.percentile(np.abs(das_array), pclip)

# Plot all channels as a heatmap
plt.figure(figsize=(12, 6))
plt.imshow(das_array, aspect='auto',
           extent=[t_axis[0], t_axis[-1], x_axis[-1], x_axis[0]],
           vmin=-das_clip, vmax=das_clip,
           cmap='seismic')
# plt.xlim([0, 60]) # 1 min

plt.colorbar(label='Strain Rate Amplitude')
plt.xlabel('Time [s]')
plt.ylabel('Distance along bridge [m]')
plt.title(f'Bridge DAS: Space-Time Wavefield ({das_paths[0].name})')
plt.tight_layout()
plt.show()

In [ ]:
# Distance Calculations
ch_400m = int(400 / dx) # result: 163
ch_600m = int(600 / dx) # result: 244

plot_das_wavefield(
    das_array,
    fs=fs,
    dx=dx,
    start_sec=0,
    duration_sec=60,    # Zoom in to 1 minute 
    start_chan=None,#ch_400m,
    end_chan=None,#ch_600m,
    pclip=98,            
    title="Gyeonbu Expressway", 
    figsize=(12, 6)
)

In [ ]:
ch_400m = int(400 / dx)  # 163
ch_600m = int(600 / dx)  # 244

plot_das_psd(
    data=das_array,
    fs=fs,
    dx=dx,
    start_chan=ch_400m,
    end_chan=ch_600m,
    nperseg=4096,           # ~16-second windows (excellent frequency resolution for 250 Hz data)
    xlim=(0.0, 50.0),       
    ylim=(30, 70.0),              
    xscale="linear",        # Linear is essential for picking out harmonic bridge modes
    ylabel="PSD (dB)",
    figsize=(12, 6)
)

### Bridge DAS Processing Part II

In [ ]:
# 1. Define Bridge-Specific Parameters
f_sag_min, f_sag_max = 0.1, 1.0    # Quasi-static band (Truck weight / Sag)
f_dyn_min, f_dyn_max = 2.0, 50.0   # Dynamic band (Structural Ringing / Guided Waves)
p_val = 99                         # Percentile for clipping outliers

# Define exact channel boundaries for the suspended deck
ch_400m = int(400 / dx)
ch_600m = int(600 / dx)

# 2. Filter the FULL array to maintain global channel indices
das_sag = bandpass_filter_tukey(das_array, fs=fs, f1=f_sag_min, f2=f_sag_max, alpha=0.05, order=4)
das_dyn = bandpass_filter_tukey(das_array, fs=fs, f1=f_dyn_min, f2=f_dyn_max, alpha=0.05, order=4)

In [ ]:
# 3. Plot Branch A: Quasi-Static Sag (0.1 - 1.0 Hz)
plot_das_wavefield(
    data=das_sag,
    fs=fs,
    dx=dx,
    start_sec=10,
    duration_sec=60*2,        
    start_chan=None, #ch_400m,
    end_chan=None, #ch_600m,
    pclip=p_val,
    cmap="seismic",
    clabel="Strain Rate Amplitude",
    title=f"Quasi-Static Deformation ({f_sag_min}-{f_sag_max} Hz) | 'Truck Sag'"
)

In [ ]:
# 4. Plot Branch B: Dynamic Ringing (1.0 - 50.0 Hz)
plot_das_wavefield(
    data=das_dyn,
    fs=fs,
    dx=dx,
    start_sec=10,
    duration_sec=10,       
    start_chan=None, #ch_400m,
    end_chan=None,#ch_600m,
    pclip=p_val,
    cmap="seismic",
    clabel="Strain Rate Amplitude",
    title=f"Dynamic Guided Waves ({f_dyn_min}-{f_dyn_max} Hz) | 'Structural Ringing'"
)

In [ ]:
ch_400m = int(400 / dx)  # 163
ch_600m = int(600 / dx)  # 244

plot_das_psd(
    data=das_dyn,
    fs=fs,
    dx=dx,
    start_chan=ch_400m,
    end_chan=ch_600m,
    nperseg=4096,           
    xlim=(0.0, 50.0),       
    ylim=(-30.0, 70.0),               
    xscale="linear",         
    ylabel="PSD (dB)",
    figsize=(12, 6)
)

In [ ]:
# Define bridge boundaries
ch_400m = int(400 / dx)
ch_600m = int(600 / dx)

# Initialize an empty array to hold the cumulative PSD
cumulative_psd = None
num_files = 0

for p in bridge_files[:8]:  # Start with just 4 to test
    print(f"Processing: {p.name}")
    
    # 1. Load the data and slice to the bridge
    data = np.load(p)['data'] # Adjust key if needed based on your npz structure
    bridge_slice = data[ch_400m:ch_600m, :]
    
    # 2. Filter out the truck sag (1.0 to 50.0 Hz)
    filtered_slice = bandpass_filter_tukey(bridge_slice, fs=fs, f1=1.0, f2=50.0)
    
    # 3. Compute PSD for this specific file
    freqs, psd_values = signal.welch(filtered_slice, fs=fs, nperseg=4096, axis=1)
    file_mean_psd = np.mean(psd_values, axis=0)
    
    # 4. Add to the cumulative total
    if cumulative_psd is None:
        cumulative_psd = np.zeros_like(file_mean_psd)
        
    cumulative_psd += file_mean_psd
    num_files += 1

# 5. Compute the final master PSD
master_psd = cumulative_psd / num_files
master_psd_db = 10 * np.log10(master_psd + 1e-12)

In [ ]:
# 6. Plotting the Master Stacked PSD
plt.figure(figsize=(14, 6))

# Plot the 1D arrays directly
plt.plot(freqs, master_psd_db, color='black', linewidth=1)

# Lock the X-axis to the dynamic structural band
plt.xlim(0.0, 50.0) 
plt.ylim(-30.0, 70.0)

# Formatting
plt.title(f"Ensemble Averaged PSD ({num_files} files stacked | 0.40 - 0.60 km)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD (dB)")

# Add a light grid to make picking exact frequencies easier
plt.grid(True, which="both", ls="-", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# 1. Grab all files
bridge_dir = Path('..') / 'data' / 'raw_bridge'
all_files = sorted(bridge_dir.glob('*_bridge.npz'))

# 2. Initialize a dictionary to act as our "Hourly Buckets"
# Format: { "2025-07-22_02": [file1, file2, ...], ... }
hourly_buckets = defaultdict(list)

# 3. Parse the filenames and sort them
for p in all_files:
    # Extract the timestamp part: "20250722_025000"
    filename_parts = p.stem.split('_')
    date_str = filename_parts[0]  # "20250722"
    time_str = filename_parts[1]  # "025000"
    
    # Parse into a real datetime object
    dt = datetime.strptime(f"{date_str}_{time_str}", "%Y%m%d_%H%M%S")
    
    # Create a unique string for that specific hour (e.g., "2025-07-22 02:00")
    hour_key = dt.strftime("%Y-%m-%d %H:00")
    
    # Drop the file into the correct bucket
    hourly_buckets[hour_key].append(p)

print(f"Sorted {len(all_files)} files into {len(hourly_buckets)} hourly buckets.")

In [ ]:
import numpy as np
from scipy import signal

# 1. Define bridge boundaries
ch_400m = int(400 / dx)
ch_600m = int(600 / dx)

# 2. Initialize storage for our 2D Matrix
spectrogram_rows = []
valid_time_keys = []
freqs = None  # We will save the frequency axis on the first pass

print(f"Starting PSD extraction for {len(hourly_buckets)} hours...\n")

# 3. Loop through each hourly bucket
for hour_key, file_list in hourly_buckets.items():
    print(f"Processing {hour_key} ({len(file_list)} files)...", end="")
    
    hour_data_list = []
    
    # A. Load and Slice
    for p in file_list:
        npz_file = np.load(p)
        arr = npz_file[npz_file.files[0]]
        
        # SLICE IMMEDIATELY: Only keep the 400m - 600m suspended deck
        bridge_slice = arr[ch_400m:ch_600m, :]
        hour_data_list.append(bridge_slice)
        
    # B. Stack the hour
    if not hour_data_list:
        print(" Skipped (Empty).")
        continue
    stacked_hour = np.concatenate(hour_data_list, axis=1)
    
    # C. Filter out the truck sag (1.0 to 50.0 Hz)
    # Using alpha=0.01 since the array is very long (a full hour)
    filtered_hour = bandpass_filter_tukey(stacked_hour, fs=fs, f1=1.0, f2=50.0, alpha=0.01, order=4)
    
    # D. Compute Welch PSD
    f, psd_values = signal.welch(filtered_hour, fs=fs, nperseg=4096, axis=1)
    
    # Average across all channels to get the single "Master" curve for this hour
    mean_psd = np.mean(psd_values, axis=0)
    
    # Convert to dB
    psd_db = 10 * np.log10(mean_psd + 1e-12)
    
    # E. Store the results
    spectrogram_rows.append(psd_db)
    valid_time_keys.append(hour_key)
    
    if freqs is None:
        freqs = f
        
    print(" Done.")

# 4. Finalize the Matrix
spectrogram_matrix = np.array(spectrogram_rows)

print("\n--- Processing Complete ---")
print(f"Matrix Shape: {spectrogram_matrix.shape} (Hours x Frequencies)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# 1. Setup the Figure and Axis
fig, ax = plt.subplots(figsize=(12, 6))

# Initialize the plot with the very first hour (Row 0)
# We use freqs and the first row of your matrix
line, = ax.plot(freqs, spectrogram_matrix[0, :], color='black', linewidth=1.5)

# Formatting the static layout
ax.set_xlim(0.0, 50.0)      # Lock to the structural band
ax.set_ylim(0, np.max(spectrogram_matrix) + 5) # Dynamically lock Y-axis to max dB
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("PSD (dB)")
ax.grid(True, which="both", ls="-", alpha=0.2)

# 2. Define the Animation Update Function
def animate(frame_index):
    """Updates the line data and title for each frame (hour)."""
    # Grab the specific row from the 2D matrix
    new_y_data = spectrogram_matrix[frame_index, :]
    line.set_ydata(new_y_data)
    
    # Update title with the timestamp
    current_time = valid_time_keys[frame_index]
    ax.set_title(f"Dynamic Bridge Resonances | {current_time} (Hour {frame_index + 1}/{len(valid_time_keys)})")
    
    return line,

# 3. Create the Animation Object
ani = FuncAnimation(
    fig, 
    animate, 
    frames=spectrogram_matrix.shape[0], 
    interval=200,   # 200 milliseconds per frame (5 FPS)
    blit=True
)

# 4. Prevent the static double-plot
plt.close(fig) 

# 5. Render directly in the Notebook
display(HTML(ani.to_jshtml()))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft
from scipy.linalg import svd

def run_fdd_for_target(bridge_slice, fs, dx, start_km, target_freq, nperseg=4096):
    """
    Extracts the physical mode shape of a bridge at a specific target frequency using FDD.
    """
    print(f"Running FDD for Target Frequency: {target_freq} Hz...")
    
    # 1. Compute the STFT to get complex frequency data across multiple time windows
    # Zxx shape: (Channels, Frequencies, Time Windows)
    f, t, Zxx = stft(bridge_slice, fs=fs, nperseg=nperseg, axis=1)
    
    # 2. Find the exact frequency bin closest to your target
    freq_idx = np.argmin(np.abs(f - target_freq))
    actual_freq = f[freq_idx]
    print(f"Locked onto closest frequency bin: {actual_freq:.3f} Hz")
    
    # 3. Extract the complex spatial array for just that frequency
    # Shape: (Channels, Time Windows)
    Y = Zxx[:, freq_idx, :]
    
    # 4. Build the Cross-Spectral Density (CSD) Matrix
    # We multiply the matrix by its complex conjugate transpose and average over time
    num_windows = Y.shape[1]
    G = np.dot(Y, Y.conj().T) / num_windows
    
    # 5. Singular Value Decomposition (SVD)
    # U contains the singular vectors (Mode Shapes)
    # S contains the singular values (Energy)
    U, S, Vh = svd(G)
    
    # 6. Extract the Fundamental Mode Shape
    # The first column of U represents the dominant physical shape at this frequency
    mode_shape = U[:, 0]
    
    # Force the maximum deflection to be positive for consistent visualization
    if np.real(mode_shape[np.argmax(np.abs(mode_shape))]) < 0:
        mode_shape *= -1
        
    # --- Plotting the Mode Shape ---
    num_channels = bridge_slice.shape[0]
    distance_axis = start_km + (np.arange(num_channels) * dx / 1000.0)
    
    plt.figure(figsize=(10, 5))
    
    # We plot the real part of the complex mode shape
    plt.plot(distance_axis, np.real(mode_shape), color='blue', linewidth=2, marker='o', markersize=4)
    
    # Visual aides
    plt.axhline(0, color='black', linewidth=1, linestyle='--')
    plt.fill_between(distance_axis, 0, np.real(mode_shape), color='blue', alpha=0.1)
    
    plt.title(f"Extracted Mode Shape | Frequency: {actual_freq:.2f} Hz")
    plt.xlabel("Distance along bridge [km]")
    plt.ylabel("Normalized Amplitude")
    plt.grid(True, alpha=0.3)
    
    # Set the x-axis to exactly match the bridge span
    plt.xlim(distance_axis[0], distance_axis[-1])
    
    plt.tight_layout()
    plt.show()
    
    return mode_shape, actual_freq

In [ ]:
# --- How to call it ---
# Pass in your fully stacked, filtered hourly array (or even just 5 minutes of busy traffic)
# Let's extract that strong peak you saw around 11.0 Hz
extracted_mode, f_exact = run_fdd_for_target(
    bridge_slice=filtered_hour, # Use one of your filtered hourly arrays from the loop
    fs=fs, 
    dx=dx, 
    start_km=0.400, 
    target_freq=7.8 
)

In [ ]:
# --- How to call it ---
# Pass in your fully stacked, filtered hourly array (or even just 5 minutes of busy traffic)
# Let's extract that strong peak you saw around 11.0 Hz
extracted_mode, f_exact = run_fdd_for_target(
    bridge_slice=filtered_hour, # Use one of your filtered hourly arrays from the loop
    fs=fs, 
    dx=dx, 
    start_km=0.400, 
    target_freq=11
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft
from scipy.linalg import svd

def plot_fdd_singular_values(bridge_slice, fs, nperseg=4096, max_freq=20.0):
    """
    Generates the classic FDD Singular Value vs Frequency plot for peak picking.
    """
    print("Computing STFT across all channels...")
    # 1. Compute STFT (Channels, Frequencies, Time Windows)
    f, t, Zxx = stft(bridge_slice, fs=fs, nperseg=nperseg, axis=1)
    
    # Restrict to our frequency of interest (e.g., 0 to 20 Hz like your image)
    freq_mask = f <= max_freq
    f_band = f[freq_mask]
    Zxx_band = Zxx[:, freq_mask, :]
    
    sv1 = np.zeros(len(f_band))
    sv2 = np.zeros(len(f_band))
    
    print("Performing SVD at every frequency bin...")
    # 2. Loop through every single frequency bin
    for i in range(len(f_band)):
        # Extract complex spatial array for this frequency across all time windows
        Y = Zxx_band[:, i, :]
        
        # Build Cross-Spectral Density (CSD) Matrix
        num_windows = Y.shape[1]
        G = np.dot(Y, Y.conj().T) / num_windows
        
        # Perform SVD
        U, S, Vh = svd(G)
        
        # Store the first two singular values (Energy)
        sv1[i] = S[0]
        sv2[i] = S[1]
        
    # 3. Convert to decibels (dB) for visualization
    sv1_db = 10 * np.log10(sv1 + 1e-12)
    sv2_db = 10 * np.log10(sv2 + 1e-12)
    
    # 4. Plotting (Replicating your MATLAB image)
    fig, ax = plt.subplots(figsize=(10, 5))
    
    ax.plot(f_band, sv1_db, color='#0072BD', linewidth=1.5, label='1st Singular Value (SV1)')
    ax.plot(f_band, sv2_db, color='black', linewidth=1.0, label='2nd Singular Value (SV2)')
    
    ax.set_title("FDD Peak Picking: Singular Values vs Frequency")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Singular Values (dB)")
    
    # Match the X-axis limit of your MATLAB plot
    ax.set_xlim(0, max_freq)
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    plt.show()

# --- How to call it ---
plot_fdd_singular_values(filtered_hour, fs=fs, max_freq=20.0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft
from scipy.linalg import svd

def run_fdd_with_muting(bridge_slice, fs, dx, start_km, target_freq, nperseg=4096, mute_spans=None):
    """
    Extracts mode shapes using FDD, with the ability to spatially mute 'screaming' channels.
    mute_spans: List of tuples representing km ranges to mute, e.g., [(0.480, 0.490), (0.510, 0.525)]
    """
    num_channels = bridge_slice.shape[0]
    distance_axis = start_km + (np.arange(num_channels) * dx / 1000.0)
    
    # --- SURGICAL MUTING ---
    # We create a copy so we don't permanently delete the data in your main array
    clean_slice = bridge_slice.copy()
    
    if mute_spans is not None:
        for (m_start, m_end) in mute_spans:
            # Find the indices that fall inside the bad km range
            bad_idx = np.where((distance_axis >= m_start) & (distance_axis <= m_end))[0]
            # Force those channels to perfectly zero
            clean_slice[bad_idx, :] = 0.0
            print(f"Muted {len(bad_idx)} channels between {m_start} and {m_end} km.")

    # 1. Compute the STFT on the cleaned data
    f, t, Zxx = stft(clean_slice, fs=fs, nperseg=nperseg, axis=1)
    
    # 2. Find exact frequency
    freq_idx = np.argmin(np.abs(f - target_freq))
    actual_freq = f[freq_idx]
    
    # 3. Extract complex spatial array
    Y = Zxx[:, freq_idx, :]
    
    # 4. Build CSD Matrix
    G = np.dot(Y, Y.conj().T) / Y.shape[1]
    
    # 5. SVD
    U, S, Vh = svd(G)
    mode_shape = U[:, 0]
    
    # Force positive peak for consistency
    if np.real(mode_shape[np.argmax(np.abs(mode_shape))]) < 0:
        mode_shape *= -1
        
    # --- Plotting ---
    plt.figure(figsize=(10, 5))
    plt.plot(distance_axis, np.real(mode_shape), color='blue', linewidth=2, marker='o', markersize=4)
    plt.axhline(0, color='black', linewidth=1, linestyle='--')
    
    # Highlight the muted zones in red so you remember they are blind spots
    if mute_spans is not None:
        for (m_start, m_end) in mute_spans:
            plt.axvspan(m_start, m_end, color='red', alpha=0.2, label='Muted Zone')
            
    plt.title(f"Extracted Mode Shape (Cleaned) | Frequency: {actual_freq:.2f} Hz")
    plt.xlabel("Distance along bridge [km]")
    plt.ylabel("Normalized Amplitude")
    plt.grid(True, alpha=0.3)
    plt.xlim(distance_axis[0], distance_axis[-1])
    
    # Prevent duplicate labels in legend
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    if by_label:
        plt.legend(by_label.values(), by_label.keys())
        
    plt.tight_layout()
    plt.show()
    
    return mode_shape, actual_freq

# --- Run the Fix ---
extracted_mode, f_exact = run_fdd_with_muting(
    bridge_slice=filtered_hour, 
    fs=fs, 
    dx=dx, 
    start_km=0.400, 
    target_freq=7.81,
    mute_spans=[(0.480, 0.490), (0.510, 0.525)]  # Muting the two massive spikes
)